# Fracture Classifier v2 — Training (Colab)
**Target**: AUC ≥ 0.95 | Architecture: ConvNeXt-Small (ImageNet-22K)

**Datasets**: GRAZPEDWRI-DX + FracAtlas + BoneFractureCVProject + YOLOBoneFracture

**GPU**: Auto-detects VRAM → adjusts batch size accordingly

In [ ]:
# ── Cell 1: GPU & Environment Check ──────────────────────────────────────────
import subprocess, sys
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")

if not torch.cuda.is_available():
    print("\n⚠️  GPU олдсонгүй — Runtime → Change runtime type → GPU сонгоно уу")
else:
    name  = torch.cuda.get_device_name(0)
    vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  : {name}")
    print(f"VRAM : {vram:.0f} GB")

    # VRAM-д тохируулсан batch size
    if vram >= 40:
        BATCH_SIZE, IMG_SIZE = 96, 384
    elif vram >= 24:
        BATCH_SIZE, IMG_SIZE = 64, 320
    elif vram >= 16:
        BATCH_SIZE, IMG_SIZE = 32, 256
    else:
        BATCH_SIZE, IMG_SIZE = 16, 224

    print(f"→ Auto batch={BATCH_SIZE}  img={IMG_SIZE}x{IMG_SIZE}")

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────────────────
!pip install -q timm albumentations scikit-learn tqdm pandas pillow kaggle

In [ ]:
# ── Cell 3: Google Drive mount + Kaggle setup ─────────────────────────────────
from google.colab import drive, files
drive.mount('/content/drive')

import os
from pathlib import Path

# Drive дотор хадгалах хавтас
DRIVE_DIR = Path('/content/drive/MyDrive/fracture_classifier_v2')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR  = DRIVE_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
DATA_DIR  = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

print(f"Checkpoints → {CKPT_DIR}")
print(f"Data        → {DATA_DIR}")

# Kaggle API тохируулах
# kaggle.json файлаа Drive-д байрлуулсан байх ёстой:
#   /content/drive/MyDrive/kaggle.json
kaggle_json = Path('/content/drive/MyDrive/kaggle.json')
if kaggle_json.exists():
    os.makedirs('/root/.kaggle', exist_ok=True)
    import shutil
    shutil.copy(kaggle_json, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print("✓ Kaggle API тохируулагдлаа")
else:
    print("⚠️  kaggle.json олдсонгүй — гараар upload хийх шаардлагатай")
    print("   → Drive-д /content/drive/MyDrive/kaggle.json байрлуулна уу")

In [ ]:
# ── Cell 4: Imports + Config ──────────────────────────────────────────────────
import os, json, time, gc, random, math, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_recall_curve, f1_score,
    confusion_matrix, classification_report
)
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
class Config:
    PROJECT_ROOT = DRIVE_DIR
    DATA_ROOT    = DATA_DIR
    CKPT_DIR     = CKPT_DIR

    KAGGLE_DATASETS = {
        'grazped' : 'jillannahmed/grazpedwri-dx',
        'bfcv'    : 'pkdarabi/bone-fracture-detection-computer-vision-project',
        'yolo_frac': 'deepakat002/yolo-object-detection-data-bone-fracture',
    }

    ENCODER      = 'convnext_small.in22k'  # ImageNet-22K pretrained
    DROP_RATE    = 0.3
    NUM_CLASSES  = 1

    # VRAM-д тохируулсан (Cell 1-д тодорхойлогдсон)
    IMG_SIZE    = IMG_SIZE
    BATCH_SIZE  = BATCH_SIZE
    NUM_WORKERS = 2
    PIN_MEMORY  = True
    USE_AMP     = True
    COMPILE     = False  # Colab-д torch.compile удаан → OFF
    CHANNELS_LAST = True

    STAGE1_EPOCHS = 5
    STAGE2_EPOCHS = 30
    EPOCHS        = STAGE1_EPOCHS + STAGE2_EPOCHS

    LR_HEAD_S1   = 1e-3
    LR_ENC_S2    = 5e-6
    LR_HEAD_S2   = 1e-4
    WEIGHT_DECAY = 1e-4
    WARMUP_EPOCHS = 2

    LABEL_SMOOTH  = 0.05
    MIXUP_ALPHA   = 0.2
    FOCAL_GAMMA   = 2.0
    GRAD_CLIP     = 1.0
    SEED          = 42
    TARGET_RECALL = 0.90

cfg = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
random.seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)
    torch.backends.cudnn.benchmark = True

print(f"Device : {DEVICE}")
print(f"Model  : {cfg.ENCODER}")
print(f"IMG    : {cfg.IMG_SIZE}x{cfg.IMG_SIZE}  Batch: {cfg.BATCH_SIZE}")

In [ ]:
# ── Cell 5: Dataset Download & CSV Preparation ───────────────────────────────
def download_kaggle(slug, dest):
    dest = Path(dest)
    if dest.exists() and any(dest.iterdir()):
        print(f'  [SKIP] {slug}')
        return
    dest.mkdir(parents=True, exist_ok=True)
    print(f'  Татаж байна: {slug} ...')
    os.system(f'kaggle datasets download {slug} --path {dest} --unzip -q')
    n = sum(len(fs) for _,_,fs in os.walk(dest))
    print(f'  → {n} файл')


def _parse_grazped(root, rows):
    root = Path(root)
    csv_files = list(root.rglob('*.csv'))
    if not csv_files: return
    df = pd.read_csv(csv_files[0])
    img_col = next((c for c in df.columns if 'file' in c.lower()), None)
    lab_col = next((c for c in df.columns
                    if 'fracture' in c.lower() or 'label' in c.lower()), None)
    if not img_col or not lab_col: return
    img_dir = next((p for p in root.rglob('images') if p.is_dir()), root)
    n = 0
    for _, row in df.iterrows():
        path = img_dir / str(row[img_col])
        if path.exists():
            label = 1 if str(row[lab_col]).strip().lower() in ('1','yes','fracture','true') else 0
            rows.append({'path': str(path), 'label': label, 'source': 'grazped'})
            n += 1
    print(f'  [GRAZPED] {n} зураг')


def _parse_yolo_folder(root, rows, source):
    root = Path(root)
    img_dirs = list(root.rglob('images')) or [root]
    n_pos = n_neg = 0
    for img_dir in img_dirs:
        for ip in img_dir.rglob('*'):
            if ip.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
            lp = ip.with_suffix('.txt')
            if not lp.exists():
                lp = ip.parent.parent / 'labels' / ip.with_suffix('.txt').name
            if lp.exists() and lp.stat().st_size > 3:
                rows.append({'path': str(ip), 'label': 1, 'source': source})
                n_pos += 1
            else:
                rows.append({'path': str(ip), 'label': 0, 'source': source})
                n_neg += 1
    print(f'  [{source}] fractured={n_pos}  neg={n_neg}')


def prepare_data():
    csv_path = cfg.PROJECT_ROOT / 'train.csv'
    if csv_path.exists():
        print('CSV байна — татахгүй')
        train_df = pd.read_csv(cfg.PROJECT_ROOT / 'train.csv')
        val_df   = pd.read_csv(cfg.PROJECT_ROOT / 'val.csv')
        test_df  = pd.read_csv(cfg.PROJECT_ROOT / 'test.csv')
        return train_df, val_df, test_df

    print('\n' + '='*55)
    print('DATA DOWNLOAD & PREPARATION')
    print('='*55)
    rows = []

    # FracAtlas: Drive-д байвал ашиглана
    fa_drive = DRIVE_DIR / 'fracatlas'
    if fa_drive.exists():
        print(f'\n[FracAtlas] {fa_drive}')
        for split in ['train','val','test']:
            for p in (fa_drive / split / 'img').glob('*.jpg'):
                rows.append({'path': str(p), 'label': 1, 'source': 'fracatlas'})
        nf_dir = fa_drive / 'not fractured' / 'img'
        for p in nf_dir.glob('*.jpg'):
            rows.append({'path': str(p), 'label': 0, 'source': 'fracatlas_neg'})
        n_fa = sum(1 for r in rows if 'fracatlas' in r['source'])
        print(f'  → {n_fa} зураг')
    else:
        print('[FracAtlas] Drive-д олдсонгүй — Skip')
        print('  → Drive-д fracture_classifier_v2/fracatlas/ хавтас байрлуулна уу')

    # Kaggle datasets
    graz_dir = cfg.DATA_ROOT / 'grazped'
    download_kaggle(cfg.KAGGLE_DATASETS['grazped'], graz_dir)
    _parse_grazped(graz_dir, rows)

    bfcv_dir = cfg.DATA_ROOT / 'bfcv'
    download_kaggle(cfg.KAGGLE_DATASETS['bfcv'], bfcv_dir)
    _parse_yolo_folder(bfcv_dir, rows, 'bfcv')

    yolo_dir = cfg.DATA_ROOT / 'yolo_frac'
    download_kaggle(cfg.KAGGLE_DATASETS['yolo_frac'], yolo_dir)
    _parse_yolo_folder(yolo_dir / 'data' / 'training', rows, 'yolo')

    df = pd.DataFrame(rows).sample(frac=1, random_state=cfg.SEED).reset_index(drop=True)
    cnt = Counter(df['label'])
    print(f'\nНийт: {len(df)}  fractured={cnt[1]}  not_frac={cnt[0]}')
    print(f'Харьцаа 1:{cnt[0]/max(cnt[1],1):.1f}')

    fa_test  = df[df['source'] == 'fracatlas'].sample(frac=0.1, random_state=cfg.SEED)
    rest     = df.drop(fa_test.index)
    val_df   = rest.sample(n=int(0.1 * len(rest)), random_state=cfg.SEED)
    train_df = rest.drop(val_df.index)

    train_df.to_csv(cfg.PROJECT_ROOT / 'train.csv', index=False)
    val_df.to_csv(cfg.PROJECT_ROOT / 'val.csv',   index=False)
    fa_test.to_csv(cfg.PROJECT_ROOT / 'test.csv', index=False)
    print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(fa_test)}')
    return train_df, val_df, fa_test


train_df, val_df, test_df = prepare_data()

In [ ]:
# ── Cell 6: Dataset + DataLoader ─────────────────────────────────────────────
def get_transforms(split):
    if split == 'train':
        return A.Compose([
            A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
            A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.1),
            A.Rotate(limit=20, border_mode=0, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.15,
                               rotate_limit=0, border_mode=0, p=0.4),
            A.ElasticTransform(alpha=1, sigma=50, p=0.3),
            A.RandomBrightnessContrast(0.25, 0.25, p=0.6),
            A.CoarseDropout(max_holes=10, max_height=32,
                            max_width=32, fill_value=0, p=0.3),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])


class FractureDataset(Dataset):
    def __init__(self, df, split='train'):
        self.df = df.reset_index(drop=True)
        self.tf = get_transforms(split)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(row['path']).convert('RGB'))
        img = self.tf(image=img)['image']
        return img, torch.tensor(float(row['label']), dtype=torch.float32)


def make_weighted_sampler(df):
    labels = df['label'].values
    cnt = Counter(labels)
    w = {0: 1.0/cnt[0], 1: 1.0/cnt[1]}
    weights = [w[l] for l in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


def make_loader(df, split, sampler=None):
    ds = FractureDataset(df, split)
    return DataLoader(
        ds,
        batch_size=cfg.BATCH_SIZE,
        sampler=sampler,
        shuffle=(sampler is None and split == 'train'),
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        drop_last=(split == 'train'),
        persistent_workers=(cfg.NUM_WORKERS > 0),
    )


cnt      = Counter(train_df['label'])
pos_wt   = torch.tensor([cnt[0] / max(cnt[1], 1)], dtype=torch.float32)
sampler  = make_weighted_sampler(train_df)
train_loader = make_loader(train_df, 'train', sampler=sampler)
val_loader   = make_loader(val_df,   'val')
test_loader  = make_loader(test_df,  'test')

print(f'Train batches: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')
print(f'pos_weight: {pos_wt.item():.2f}')

In [ ]:
# ── Cell 7: Model + Loss ──────────────────────────────────────────────────────
class FractureClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            cfg.ENCODER, pretrained=True, num_classes=0, drop_rate=cfg.DROP_RATE)
        feat_dim = self.encoder.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(cfg.DROP_RATE),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(cfg.DROP_RATE * 0.5),
            nn.Linear(256, cfg.NUM_CLASSES),
        )
    def forward(self, x):
        return self.head(self.encoder(x))
    def freeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad = True


class FocalBCELoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None, label_smooth=0.05):
        super().__init__()
        self.gamma = gamma
        self.pw    = pos_weight
        self.smooth = label_smooth
    def forward(self, logits, targets):
        t = targets * (1 - self.smooth) + 0.5 * self.smooth
        pw = self.pw.to(logits.device) if self.pw is not None else None
        bce = F.binary_cross_entropy_with_logits(logits, t, pos_weight=pw, reduction='none')
        pt  = torch.exp(-bce)
        return ((1 - pt) ** self.gamma * bce).mean()


def mixup_data(x, y, alpha=0.2):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(loss_fn, pred, ya, yb, lam):
    return lam * loss_fn(pred, ya) + (1 - lam) * loss_fn(pred, yb)


model   = FractureClassifier().to(DEVICE)
if cfg.CHANNELS_LAST:
    model = model.to(memory_format=torch.channels_last)
loss_fn = FocalBCELoss(cfg.FOCAL_GAMMA, pos_wt, cfg.LABEL_SMOOTH)
scaler  = GradScaler(enabled=cfg.USE_AMP)

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model params: {total:.1f}M')
print(f'Encoder: {cfg.ENCODER}')

In [ ]:
# ── Cell 8: Train / Validate helpers ─────────────────────────────────────────
def get_scheduler(optimizer, total_ep, warmup_ep):
    def lr_lambda(ep):
        if ep < warmup_ep: return (ep + 1) / warmup_ep
        prog = (ep - warmup_ep) / (total_ep - warmup_ep)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, loader, optimizer, scaler, loss_fn, epoch):
    model.train()
    total_loss = 0; n = 0
    pbar = tqdm(loader, desc=f'Ep{epoch+1:02d} train', leave=False)
    for x, y in pbar:
        if cfg.CHANNELS_LAST:
            x = x.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)
        else:
            x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE)
        x, ya, yb, lam = mixup_data(x, y, cfg.MIXUP_ALPHA)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(x).squeeze(1)
            loss   = mixup_loss(loss_fn, logits, ya, yb, lam)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], cfg.GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item(); n += 1
        pbar.set_postfix(loss=f'{total_loss/n:.4f}')
    return total_loss / n


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        if cfg.CHANNELS_LAST:
            x = x.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)
        else:
            x = x.to(DEVICE, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(x).squeeze(1)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(y.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0

    prec_arr, rec_arr, thr_arr = precision_recall_curve(labels, probs)
    mask = rec_arr[:-1] >= cfg.TARGET_RECALL
    best_thr = float(thr_arr[mask][np.argmax(prec_arr[:-1][mask])]) if mask.any() else 0.5
    preds = (probs >= best_thr).astype(int)
    return {
        'auc': float(auc), 'threshold': best_thr,
        'accuracy': float(accuracy_score(labels, preds)),
        'f1': float(f1_score(labels, preds, zero_division=0)),
        'probs': probs, 'labels': labels,
    }


def _save(model, optimizer, epoch, metrics, name):
    state = (model._orig_mod.state_dict()
             if hasattr(model, '_orig_mod') else model.state_dict())
    torch.save({
        'model': state, 'epoch': epoch + 1,
        'metrics': {k: v for k, v in metrics.items() if k not in ('probs','labels')},
        'config': {'encoder': cfg.ENCODER, 'img_size': cfg.IMG_SIZE},
    }, cfg.CKPT_DIR / name)
    print(f'  ✓ Хадгалагдлаа: {name}  (AUC={metrics["auc"]:.4f})')


history = []
best_auc = 0.0
print('Helpers тодорхойлогдлоо ✓')

In [ ]:
# ── Cell 9: Stage 1 — Head warming (encoder хөлдөөж) ────────────────────────
print('='*55)
print(f'STAGE 1 — Head ({cfg.STAGE1_EPOCHS} epoch)  LR={cfg.LR_HEAD_S1:.0e}')
print('='*55)

model.freeze_encoder()
optimizer1 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=cfg.LR_HEAD_S1, weight_decay=cfg.WEIGHT_DECAY)
scheduler1 = get_scheduler(optimizer1, cfg.STAGE1_EPOCHS, 1)

for epoch in range(cfg.STAGE1_EPOCHS):
    t0 = time.time()
    tr_loss = train_one_epoch(model, train_loader, optimizer1, scaler, loss_fn, epoch)
    vl      = validate(model, val_loader)
    scheduler1.step()
    elapsed = time.time() - t0

    mark = ''
    if vl['auc'] > best_auc:
        best_auc = vl['auc']
        _save(model, optimizer1, epoch, vl, 'best_stage1.pt')
        mark = ' ★'

    print(f'Ep{epoch+1:02d} ({elapsed:.0f}s) | loss={tr_loss:.4f} | '
          f'AUC={vl["auc"]:.4f}  F1={vl["f1"]:.3f}  '
          f'thr={vl["threshold"]:.3f}{mark}')
    history.append({'stage': 1, 'epoch': epoch+1, 'loss': tr_loss,
                    'auc': vl['auc'], 'f1': vl['f1'], 'threshold': vl['threshold']})

print(f'\nStage 1 дууслаа  — Best AUC: {best_auc:.4f}')

In [ ]:
# ── Cell 10: Stage 2 — Full fine-tune ────────────────────────────────────────
print('='*55)
print(f'STAGE 2 — Full finetune ({cfg.STAGE2_EPOCHS} epoch)')
print('='*55)

model.unfreeze_encoder()
optimizer2 = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': cfg.LR_ENC_S2},
    {'params': model.head.parameters(),    'lr': cfg.LR_HEAD_S2},
], weight_decay=cfg.WEIGHT_DECAY)
scheduler2 = get_scheduler(optimizer2, cfg.STAGE2_EPOCHS, cfg.WARMUP_EPOCHS)

for epoch in range(cfg.STAGE2_EPOCHS):
    t0 = time.time()
    tr_loss = train_one_epoch(
        model, train_loader, optimizer2, scaler, loss_fn,
        cfg.STAGE1_EPOCHS + epoch)
    vl      = validate(model, val_loader)
    scheduler2.step()
    elapsed = time.time() - t0

    mark = ''
    if vl['auc'] > best_auc:
        best_auc = vl['auc']
        _save(model, optimizer2, cfg.STAGE1_EPOCHS + epoch, vl, 'best_model.pt')
        mark = ' ★'

    ep_num = cfg.STAGE1_EPOCHS + epoch + 1
    print(f'Ep{ep_num:02d} ({elapsed:.0f}s) | loss={tr_loss:.4f} | '
          f'AUC={vl["auc"]:.4f}  F1={vl["f1"]:.3f}{mark}')
    history.append({'stage': 2, 'epoch': ep_num, 'loss': tr_loss,
                    'auc': vl['auc'], 'f1': vl['f1'], 'threshold': vl['threshold']})

    torch.cuda.empty_cache(); gc.collect()

print(f'\nStage 2 дууслаа  — Best AUC: {best_auc:.4f}')

# History хадгалах
pd.DataFrame(history).to_csv(cfg.PROJECT_ROOT / 'history.csv', index=False)
print('history.csv хадгалагдлаа')

In [ ]:
# ── Cell 11: Final Evaluation with TTA ───────────────────────────────────────
@torch.no_grad()
def predict_tta(model, df, n_aug=8):
    tta_tf = A.Compose([
        A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, p=0.5),
        A.RandomBrightnessContrast(0.1, 0.1, p=0.4),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])
    model.eval()
    paths  = df['path'].tolist()
    labels = df['label'].tolist()
    probs  = np.zeros(len(paths))
    for _ in tqdm(range(n_aug), desc='TTA'):
        for i, p in enumerate(paths):
            img = np.array(Image.open(p).convert('RGB'))
            x = tta_tf(image=img)['image'].unsqueeze(0)
            if cfg.CHANNELS_LAST:
                x = x.to(DEVICE, memory_format=torch.channels_last)
            else:
                x = x.to(DEVICE)
            with autocast(enabled=cfg.USE_AMP):
                probs[i] += torch.sigmoid(model(x).squeeze()).item()
    probs /= n_aug
    labels = np.array(labels)
    auc = roc_auc_score(labels, probs)
    prec_arr, rec_arr, thr_arr = precision_recall_curve(labels, probs)
    mask = rec_arr[:-1] >= cfg.TARGET_RECALL
    best_thr = float(thr_arr[mask][np.argmax(prec_arr[:-1][mask])]) if mask.any() else 0.5
    preds = (probs >= best_thr).astype(int)
    return {'auc': float(auc), 'threshold': best_thr,
            'accuracy': float(accuracy_score(labels, preds)),
            'f1': float(f1_score(labels, preds, zero_division=0)),
            'probs': probs, 'labels': labels, 'preds': preds}


# Best model ачаалах
ckpt_path = cfg.CKPT_DIR / 'best_model.pt'
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'])
print(f'Loaded: {ckpt_path}  (epoch {ckpt["epoch"]})')

print('\n[Val TTA]')
val_tta  = predict_tta(model, val_df)
print(f'  AUC={val_tta["auc"]:.4f}  F1={val_tta["f1"]:.3f}  '
      f'Acc={val_tta["accuracy"]:.3f}  thr={val_tta["threshold"]:.3f}')

print('\n[Test TTA]')
test_tta = predict_tta(model, test_df)
print(f'  AUC={test_tta["auc"]:.4f}  F1={test_tta["f1"]:.3f}  '
      f'Acc={test_tta["accuracy"]:.3f}')
print('\n  Classification Report:')
print(classification_report(test_tta['labels'], test_tta['preds'],
      target_names=['Not Fractured','Fractured'], digits=3))
print('  Confusion Matrix:')
print(confusion_matrix(test_tta['labels'], test_tta['preds']))

# Metrics хадгалах
with open(cfg.PROJECT_ROOT / 'final_metrics.json', 'w') as f:
    json.dump({
        'val_auc_tta':  val_tta['auc'],
        'test_auc_tta': test_tta['auc'],
        'threshold':    test_tta['threshold'],
        'encoder':      cfg.ENCODER,
        'img_size':     cfg.IMG_SIZE,
        'batch_size':   cfg.BATCH_SIZE,
    }, f, indent=2)
print('\nfinal_metrics.json хадгалагдлаа')

In [ ]:
# ── Cell 12: Training Curves Visualization ───────────────────────────────────
plt.style.use('dark_background')
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training History — Fracture Classifier v2', fontsize=14, color='white')

colors = {'s1': '#4FC3F7', 's2': '#81C784'}
for stage, grp in hist_df.groupby('stage'):
    c = colors[f's{stage}']
    label = f'Stage {stage}'
    axes[0].plot(grp['epoch'], grp['loss'], c=c, marker='o', ms=3, label=label)
    axes[1].plot(grp['epoch'], grp['auc'],  c=c, marker='o', ms=3, label=label)
    axes[2].plot(grp['epoch'], grp['f1'],   c=c, marker='o', ms=3, label=label)

axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('AUC');  axes[1].set_xlabel('Epoch')
axes[1].axhline(0.95, color='#FF7043', ls='--', lw=1.5, label='Target 0.95')
axes[2].set_title('F1');   axes[2].set_xlabel('Epoch')

for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(cfg.PROJECT_ROOT / 'training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n{"="*50}')
print(f'  BEST VAL AUC  : {best_auc:.4f}')
print(f'  TEST AUC (TTA): {test_tta["auc"]:.4f}')
print(f'  Test F1       : {test_tta["f1"]:.4f}')
print(f'  Threshold     : {test_tta["threshold"]:.4f}')
print(f'{"="*50}')

if test_tta['auc'] >= 0.95:
    print('  ✓ TARGET AUC ≥ 0.95 ХҮРЛЭЭ!')
else:
    gap = 0.95 - test_tta['auc']
    print(f'  ⚡ Target-аас {gap:.4f} доогуур — илүү epoch сургах хэрэгтэй')

In [ ]:
# ── Cell 13: Copy best_model.pt → Drive root (inference pipeline-д хялбар) ──
import shutil

src = cfg.CKPT_DIR / 'best_model.pt'
dst = Path('/content/drive/MyDrive/fracture_classifier_v2.pt')
shutil.copy(src, dst)
print(f'✓ Хуулагдлаа: {dst}')
print(f'  → inference_v2_colab.ipynb дотор MURA_CKPT замыг шинэчилнэ:')
print(f'    MURA_CKPT = "/content/drive/MyDrive/fracture_classifier_v2.pt"')
print(f'  → Inference pipeline-д загвар ачаалахдаа encoder нэрийг шалгана:')
print(f'    cfg.ENCODER = "{cfg.ENCODER}"')
print(f'    cfg.IMG_SIZE = {cfg.IMG_SIZE}')